<a href="https://colab.research.google.com/github/SANGHATI23/neurofhir-qc/blob/main/NeuroFHIR_QC_Notebook_01_FIXED.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NeuroFHIR-QC — Notebook 01  
## Synthetic FHIR R4 Foundation

**Notebook filename:** `01_NeuroFHIR_QC_Synthetic_FHIR_Foundation.ipynb`  
**Project root:** `/content/drive/MyDrive/neurofhir-qc`  
**Data policy:** synthetic FHIR records only; no real patient data  
**FHIR version:** R4

### What this notebook builds

This notebook creates the synthetic longitudinal FHIR context required for the three locked competition cases:

1. **Stable**
2. **Progression**
3. **Low confidence**

For each case, it generates:

- one `Patient`;
- one `Condition`;
- one baseline `ImagingStudy`;
- one follow-up `ImagingStudy`;
- one prior, already reviewed baseline-volume `Observation`.

It also creates a machine-readable demo-case manifest, a resource index, reference-integrity evidence, checksums, and a Notebook 01 audit report.

### What this notebook intentionally does not build

- no HAPI FHIR server connection;
- no SMART App Launch;
- no MRI download or preprocessing;
- no segmentation or current AI-derived volume;
- no QC score;
- no human-review transition;
- no transaction write-back Bundle.

Those functions belong to later notebooks. In particular, **no new AI-generated Observation is marked final here**.

### Before running

Notebook 00 must have passed its final foundation audit and be marked complete in `notebook_manifest.json`.  
Save this notebook into the project `notebooks/` directory before the final completion cell is run:

`/content/drive/MyDrive/neurofhir-qc/notebooks/01_NeuroFHIR_QC_Synthetic_FHIR_Foundation.ipynb`

In [1]:
# Cell 1 — Mount Drive and enforce the Notebook 00 completion gate

from __future__ import annotations

import csv
import hashlib
import json
import math
import re
import sys
import textwrap
import uuid
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Iterable

try:
    from google.colab import drive
except ImportError as exc:
    raise RuntimeError(
        "This notebook is designed for Google Colab. Open it in Colab and run again."
    ) from exc

drive.mount("/content/drive")

PROJECT_ROOT = Path("/content/drive/MyDrive/neurofhir-qc")
CONFIG_PATH = PROJECT_ROOT / "project_config.json"
NOTEBOOK_MANIFEST_PATH = PROJECT_ROOT / "notebook_manifest.json"
FOUNDATION_AUDIT_JSON = (
    PROJECT_ROOT / "evaluation/results/notebook_00_foundation_audit.json"
)
FOUNDATION_AUDIT_MD = PROJECT_ROOT / "docs/FOUNDATION_AUDIT.md"

REQUIRED_FOUNDATION_PATHS = [
    PROJECT_ROOT,
    CONFIG_PATH,
    NOTEBOOK_MANIFEST_PATH,
    PROJECT_ROOT / "README.md",
    PROJECT_ROOT / "LICENSE",
    PROJECT_ROOT / "CITATION.cff",
    PROJECT_ROOT / "SECURITY.md",
    PROJECT_ROOT / ".gitignore",
    PROJECT_ROOT / ".env.example",
    PROJECT_ROOT / "runtime_manifest.json",
    PROJECT_ROOT / "docs/SCOPE_LOCK.md",
    PROJECT_ROOT / "docs/BUILD_PLAN.md",
    FOUNDATION_AUDIT_JSON,
    FOUNDATION_AUDIT_MD,
]

missing_foundation_paths = [
    str(path) for path in REQUIRED_FOUNDATION_PATHS
    if not path.exists() or (path.is_file() and path.stat().st_size == 0)
]
if missing_foundation_paths:
    raise FileNotFoundError(
        "Notebook 00 is not fully evidenced. Run its final audit cell first.\n"
        + "\n".join(f" - {path}" for path in missing_foundation_paths)
    )

with CONFIG_PATH.open("r", encoding="utf-8") as handle:
    project_config = json.load(handle)

with NOTEBOOK_MANIFEST_PATH.open("r", encoding="utf-8") as handle:
    notebook_manifest = json.load(handle)

def notebook_entries(manifest: Any) -> list[dict[str, Any]]:
    if isinstance(manifest, list):
        return manifest
    if isinstance(manifest, dict):
        for key in ("notebooks", "entries", "workflow"):
            value = manifest.get(key)
            if isinstance(value, list):
                return value
    raise ValueError(
        "notebook_manifest.json does not contain a recognized notebook-entry list."
    )

def normalize_notebook_number(value: Any) -> str:
    text = str(value).strip()
    match = re.search(r"\d+", text)
    if not match:
        return text
    return match.group(0).zfill(2)

def find_notebook_entry(
    manifest: Any,
    notebook_number: str,
) -> dict[str, Any]:
    target = normalize_notebook_number(notebook_number)
    for entry in notebook_entries(manifest):
        candidate_values = [
            entry.get("number"),
            entry.get("notebook_number"),
            entry.get("id"),
            entry.get("filename"),
        ]
        if any(
            normalize_notebook_number(value) == target
            for value in candidate_values
            if value is not None
        ):
            return entry
    raise KeyError(f"Notebook {target} is missing from notebook_manifest.json.")

notebook_00_entry = find_notebook_entry(notebook_manifest, "00")
notebook_01_entry = find_notebook_entry(notebook_manifest, "01")

notebook_00_status = str(notebook_00_entry.get("status", "")).strip().lower()
accepted_complete_statuses = {"completed", "complete", "passed"}
if notebook_00_status not in accepted_complete_statuses:
    raise RuntimeError(
        "Notebook 00 must be marked completed before Notebook 01 begins. "
        f"Current manifest status: {notebook_00_status!r}"
    )

required_resources = {
    "Patient",
    "Condition",
    "ImagingStudy",
    "Observation",
    "DiagnosticReport",
    "Device",
    "Provenance",
    "Task",
    "Bundle",
}
configured_resources = set(project_config.get("core_fhir_resources", []))
if configured_resources != required_resources:
    raise AssertionError(
        "The locked core FHIR resource set has changed. "
        f"Configured={sorted(configured_resources)}"
    )

configured_cases = {
    str(value).strip().lower().replace("_", "-")
    for value in project_config.get("required_demo_cases", [])
}
if configured_cases != {"stable", "progression", "low-confidence"}:
    raise AssertionError(
        "The locked demonstration cases have changed. "
        f"Configured={sorted(configured_cases)}"
    )

if project_config.get("project_name") != "NeuroFHIR-QC":
    raise AssertionError("Unexpected project name in project_config.json.")

NOTEBOOK_NUMBER = "01"
DEFAULT_NOTEBOOK_FILENAME = "01_NeuroFHIR_QC_Synthetic_FHIR_Foundation.ipynb"
NOTEBOOK_FILENAME = notebook_01_entry.get("filename", DEFAULT_NOTEBOOK_FILENAME)
NOTEBOOK_SAVE_PATH = PROJECT_ROOT / "notebooks" / NOTEBOOK_FILENAME

print("=" * 88)
print("✅ Notebook 00 completion gate passed")
print(f"✅ Project root: {PROJECT_ROOT}")
print(f"✅ Locked FHIR resources: {len(configured_resources)}")
print(f"✅ Locked demo cases: {sorted(configured_cases)}")
print(f"📓 Manifest filename for Notebook 01: {NOTEBOOK_FILENAME}")
print(f"💾 Required saved-notebook path: {NOTEBOOK_SAVE_PATH}")
print("=" * 88)

Mounted at /content/drive
✅ Notebook 00 completion gate passed
✅ Project root: /content/drive/MyDrive/neurofhir-qc
✅ Locked FHIR resources: 9
✅ Locked demo cases: ['low-confidence', 'progression', 'stable']
📓 Manifest filename for Notebook 01: 01_NeuroFHIR_QC_Synthetic_FHIR_Foundation.ipynb
💾 Required saved-notebook path: /content/drive/MyDrive/neurofhir-qc/notebooks/01_NeuroFHIR_QC_Synthetic_FHIR_Foundation.ipynb


## Design rules used in this notebook

1. Every record is visibly marked as synthetic.
2. Patient identity is generated only for demonstration and contains no address, telephone number, or email.
3. The baseline volume is represented as a **prior reviewed result** with `Observation.status = final`.
4. The future AI-derived follow-up result is **not generated here**; when it is created later, it must begin as `preliminary`.
5. A local NeuroFHIR-QC code system is used for project-specific concepts rather than claiming unsupported LOINC or SNOMED CT conformance.
6. FHIR references are deterministic and validated before files are written.
7. Notebook 01 creates source context only; transaction write-back is deliberately deferred.

In [2]:
# Cell 2 — Shared constants and deterministic helper functions

FHIR_VERSION = "R4"
PROJECT_VERSION = str(project_config.get("version", "0.1.0"))

DEMO_IDENTIFIER_SYSTEM = (
    "https://neurofhir-qc.example.org/fhir/NamingSystem/demo-patient-id"
)
DEMO_TAG_SYSTEM = (
    "https://neurofhir-qc.example.org/fhir/CodeSystem/data-origin"
)
DEMO_CONDITION_SYSTEM = (
    "https://neurofhir-qc.example.org/fhir/CodeSystem/demo-condition"
)
DEMO_OBSERVATION_SYSTEM = (
    "https://neurofhir-qc.example.org/fhir/CodeSystem/neuro-observation"
)
DEMO_METHOD_SYSTEM = (
    "https://neurofhir-qc.example.org/fhir/CodeSystem/measurement-method"
)

SYNTHETIC_TAG = {
    "system": DEMO_TAG_SYSTEM,
    "code": "synthetic",
    "display": "Synthetic demonstration record",
}

FHIR_ID_PATTERN = re.compile(r"^[A-Za-z0-9\-.]{1,64}$")

def utc_now() -> str:
    return datetime.now(timezone.utc).replace(microsecond=0).isoformat().replace("+00:00", "Z")

def parse_fhir_datetime(value: str) -> datetime:
    return datetime.fromisoformat(value.replace("Z", "+00:00"))

def assert_fhir_id(value: str) -> str:
    if not FHIR_ID_PATTERN.fullmatch(value):
        raise ValueError(f"Invalid FHIR id: {value!r}")
    return value

def deterministic_dicom_uid(*parts: str) -> str:
    seed = "|".join(parts)
    generated = uuid.uuid5(uuid.NAMESPACE_URL, seed)
    uid = f"2.25.{generated.int}"
    if len(uid) > 64:
        raise AssertionError("Generated DICOM UID exceeds 64 characters.")
    return uid

def calculate_percent_change(
    baseline: float,
    followup: float,
) -> float:
    if baseline <= 0:
        raise ValueError("Baseline volume must be greater than zero.")
    return round(((followup - baseline) / baseline) * 100.0, 2)

def write_json(path: Path, payload: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = path.with_suffix(path.suffix + ".tmp")
    with temporary_path.open("w", encoding="utf-8") as handle:
        json.dump(payload, handle, indent=2, ensure_ascii=False)
        handle.write("\n")
    temporary_path.replace(path)

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def synthetic_meta(case_id: str) -> dict[str, Any]:
    return {
        "profile": [
            "http://hl7.org/fhir/StructureDefinition/DomainResource"
        ],
        "tag": [
            SYNTHETIC_TAG,
            {
                "system": DEMO_TAG_SYSTEM,
                "code": case_id,
                "display": f"NeuroFHIR-QC {case_id} demonstration case",
            },
        ],
    }

print("✅ Shared constants and deterministic helper functions are ready")
print(f"✅ FHIR target: {FHIR_VERSION}")
print(f"✅ Project version: {PROJECT_VERSION}")

✅ Shared constants and deterministic helper functions are ready
✅ FHIR target: R4
✅ Project version: 0.1.0


In [3]:
# Cell 3 — Define the three locked synthetic longitudinal demonstration cases

DEMO_CASES: list[dict[str, Any]] = [
    {
        "case_id": "stable",
        "patient": {
            "id": "nqc-stable-001",
            "identifier": "NQC-STABLE-001",
            "given": "Avery",
            "family": "Morgan",
            "gender": "female",
            "birth_date": "1976-04-12",
        },
        "baseline_datetime": "2026-01-15T09:00:00Z",
        "followup_datetime": "2026-04-15T09:00:00Z",
        "baseline_volume_ml": 14.2,
        "planned_followup_reference_volume_ml": 14.6,
        "planned_change_category": "stable",
        "future_qc_expectation": "high-confidence",
        "future_review_expectation": "accept",
        "scenario_description": (
            "Similar baseline and follow-up volume with a stable future segmentation."
        ),
    },
    {
        "case_id": "progression",
        "patient": {
            "id": "nqc-progression-001",
            "identifier": "NQC-PROGRESSION-001",
            "given": "Rowan",
            "family": "Lee",
            "gender": "male",
            "birth_date": "1969-09-24",
        },
        "baseline_datetime": "2026-01-20T10:30:00Z",
        "followup_datetime": "2026-04-20T10:30:00Z",
        "baseline_volume_ml": 12.5,
        "planned_followup_reference_volume_ml": 18.7,
        "planned_change_category": "progression",
        "future_qc_expectation": "high-confidence",
        "future_review_expectation": "accept",
        "scenario_description": (
            "Meaningful future volume increase with a stable future segmentation."
        ),
    },
    {
        "case_id": "low-confidence",
        "patient": {
            "id": "nqc-low-confidence-001",
            "identifier": "NQC-LOW-CONFIDENCE-001",
            "given": "Jordan",
            "family": "Patel",
            "gender": "female",
            "birth_date": "1982-02-08",
        },
        "baseline_datetime": "2026-01-25T14:00:00Z",
        "followup_datetime": "2026-04-25T14:00:00Z",
        "baseline_volume_ml": 16.1,
        "planned_followup_reference_volume_ml": 17.0,
        "planned_change_category": "indeterminate-until-qc",
        "future_qc_expectation": "manual-review-required",
        "future_review_expectation": "reject-or-correction",
        "scenario_description": (
            "A future perturbed or poor-quality scan should produce an unstable "
            "segmentation that remains preliminary."
        ),
    },
]

case_ids = [case["case_id"] for case in DEMO_CASES]
if set(case_ids) != {"stable", "progression", "low-confidence"}:
    raise AssertionError("The notebook must define exactly the three locked cases.")
if len(case_ids) != len(set(case_ids)):
    raise AssertionError("Duplicate demonstration case id detected.")

for case in DEMO_CASES:
    patient = case["patient"]
    assert_fhir_id(patient["id"])
    baseline_dt = parse_fhir_datetime(case["baseline_datetime"])
    followup_dt = parse_fhir_datetime(case["followup_datetime"])
    if not baseline_dt < followup_dt:
        raise AssertionError(
            f"Baseline must precede follow-up for case {case['case_id']}."
        )
    if float(case["baseline_volume_ml"]) <= 0:
        raise AssertionError("Baseline volume must be positive.")
    if float(case["planned_followup_reference_volume_ml"]) <= 0:
        raise AssertionError("Planned follow-up reference volume must be positive.")
    case["planned_percent_change"] = calculate_percent_change(
        float(case["baseline_volume_ml"]),
        float(case["planned_followup_reference_volume_ml"]),
    )

print("✅ Three synthetic demonstration cases defined")
for case in DEMO_CASES:
    print(
        f" - {case['case_id']}: "
        f"{case['baseline_volume_ml']} mL baseline; "
        f"planned reference change {case['planned_percent_change']:+.2f}%"
    )

✅ Three synthetic demonstration cases defined
 - stable: 14.2 mL baseline; planned reference change +2.82%
 - progression: 12.5 mL baseline; planned reference change +49.60%
 - low-confidence: 16.1 mL baseline; planned reference change +5.59%


In [4]:
# Cell 4 — FHIR R4 resource builders

def build_patient(case: dict[str, Any]) -> dict[str, Any]:
    patient = case["patient"]
    patient_id = assert_fhir_id(patient["id"])
    return {
        "resourceType": "Patient",
        "id": patient_id,
        "meta": synthetic_meta(case["case_id"]),
        "identifier": [
            {
                "use": "usual",
                "system": DEMO_IDENTIFIER_SYSTEM,
                "value": patient["identifier"],
            }
        ],
        "active": True,
        "name": [
            {
                "use": "usual",
                "family": patient["family"],
                "given": [patient["given"]],
            }
        ],
        "gender": patient["gender"],
        "birthDate": patient["birth_date"],
        "communication": [
            {
                "language": {
                    "coding": [
                        {
                            "system": "urn:ietf:bcp:47",
                            "code": "en",
                            "display": "English",
                        }
                    ],
                    "text": "English",
                },
                "preferred": True,
            }
        ],
    }

def build_condition(case: dict[str, Any]) -> dict[str, Any]:
    patient_id = case["patient"]["id"]
    condition_id = assert_fhir_id(f"condition-{case['case_id']}")
    return {
        "resourceType": "Condition",
        "id": condition_id,
        "meta": synthetic_meta(case["case_id"]),
        "clinicalStatus": {
            "coding": [
                {
                    "system": (
                        "http://terminology.hl7.org/CodeSystem/condition-clinical"
                    ),
                    "code": "active",
                    "display": "Active",
                }
            ]
        },
        "verificationStatus": {
            "coding": [
                {
                    "system": (
                        "http://terminology.hl7.org/CodeSystem/"
                        "condition-ver-status"
                    ),
                    "code": "confirmed",
                    "display": "Confirmed",
                }
            ]
        },
        "category": [
            {
                "coding": [
                    {
                        "system": (
                            "http://terminology.hl7.org/CodeSystem/"
                            "condition-category"
                        ),
                        "code": "problem-list-item",
                        "display": "Problem List Item",
                    }
                ]
            }
        ],
        "code": {
            "coding": [
                {
                    "system": DEMO_CONDITION_SYSTEM,
                    "code": "brain-tumor-or-lesion",
                    "display": "Synthetic brain tumor or lesion context",
                }
            ],
            "text": "Synthetic brain tumor or lesion context",
        },
        "subject": {"reference": f"Patient/{patient_id}"},
        "onsetDateTime": "2025-12-10T12:00:00Z",
        "recordedDate": "2025-12-10T12:00:00Z",
        "note": [
            {
                "text": (
                    "Synthetic condition context created only for the "
                    "NeuroFHIR-QC competition demonstration."
                )
            }
        ],
    }

def build_imaging_study(
    case: dict[str, Any],
    timepoint: str,
) -> dict[str, Any]:
    if timepoint not in {"baseline", "followup"}:
        raise ValueError("timepoint must be 'baseline' or 'followup'.")

    patient_id = case["patient"]["id"]
    case_id = case["case_id"]
    study_id = assert_fhir_id(f"imaging-{timepoint}-{case_id}")
    condition_id = f"condition-{case_id}"
    started = case[f"{timepoint}_datetime"]

    study_uid = deterministic_dicom_uid(
        "NeuroFHIR-QC",
        case_id,
        timepoint,
        "study",
    )
    series_uid = deterministic_dicom_uid(
        "NeuroFHIR-QC",
        case_id,
        timepoint,
        "series-1",
    )

    quality_note = (
        "Synthetic follow-up study reserved for a later controlled "
        "low-confidence perturbation."
        if case_id == "low-confidence" and timepoint == "followup"
        else "Synthetic study metadata for the locked demonstration workflow."
    )

    return {
        "resourceType": "ImagingStudy",
        "id": study_id,
        "meta": synthetic_meta(case_id),
        "identifier": [
            {
                "system": "urn:dicom:uid",
                "value": f"urn:oid:{study_uid}",
            }
        ],
        "status": "available",
        "modality": [
            {
                "system": (
                    "http://dicom.nema.org/resources/ontology/DCM"
                ),
                "code": "MR",
                "display": "Magnetic Resonance",
            }
        ],
        "subject": {"reference": f"Patient/{patient_id}"},
        "started": started,
        "numberOfSeries": 1,
        "numberOfInstances": 160,
        "reasonReference": [
            {"reference": f"Condition/{condition_id}"}
        ],
        "description": (
            f"Synthetic {timepoint} brain MRI study for the "
            f"NeuroFHIR-QC {case_id} case"
        ),
        "note": [{"text": quality_note}],
        "series": [
            {
                "uid": series_uid,
                "number": 1,
                "modality": {
                    "system": (
                        "http://dicom.nema.org/resources/ontology/DCM"
                    ),
                    "code": "MR",
                    "display": "Magnetic Resonance",
                },
                "description": "Synthetic T1-weighted brain MRI series",
                "numberOfInstances": 160,
                "started": started,
            }
        ],
    }

def build_prior_volume_observation(
    case: dict[str, Any],
) -> dict[str, Any]:
    case_id = case["case_id"]
    patient_id = case["patient"]["id"]
    observation_id = assert_fhir_id(f"prior-volume-{case_id}")
    baseline_study_id = f"imaging-baseline-{case_id}"

    return {
        "resourceType": "Observation",
        "id": observation_id,
        "meta": synthetic_meta(case_id),
        "identifier": [
            {
                "system": (
                    "https://neurofhir-qc.example.org/fhir/"
                    "NamingSystem/demo-observation-id"
                ),
                "value": f"NQC-PRIOR-VOLUME-{case_id.upper()}",
            }
        ],
        "status": "final",
        "category": [
            {
                "coding": [
                    {
                        "system": (
                            "http://terminology.hl7.org/CodeSystem/"
                            "observation-category"
                        ),
                        "code": "imaging",
                        "display": "Imaging",
                    }
                ]
            }
        ],
        "code": {
            "coding": [
                {
                    "system": DEMO_OBSERVATION_SYSTEM,
                    "code": "brain-tumor-or-lesion-volume",
                    "display": "Brain tumor or lesion volume",
                }
            ],
            "text": "Synthetic prior reviewed brain tumor or lesion volume",
        },
        "subject": {"reference": f"Patient/{patient_id}"},
        "focus": [{"reference": f"Condition/condition-{case_id}"}],
        "effectiveDateTime": case["baseline_datetime"],
        "issued": case["baseline_datetime"],
        "valueQuantity": {
            "value": float(case["baseline_volume_ml"]),
            "unit": "mL",
            "system": "http://unitsofmeasure.org",
            "code": "mL",
        },
        "method": {
            "coding": [
                {
                    "system": DEMO_METHOD_SYSTEM,
                    "code": "synthetic-prior-reviewed-volumetry",
                    "display": "Synthetic prior reviewed volumetry",
                }
            ],
            "text": "Synthetic prior human-reviewed volumetry",
        },
        "derivedFrom": [
            {"reference": f"ImagingStudy/{baseline_study_id}"}
        ],
        "note": [
            {
                "text": (
                    "This is a synthetic prior human-reviewed historical "
                    "measurement. It is not the current AI-generated result."
                )
            }
        ],
    }

print("✅ FHIR R4 resource builders defined")

✅ FHIR R4 resource builders defined


In [5]:
# Cell 5 — Generate Patient, Condition, ImagingStudy, and prior Observation resources

generated_utc = utc_now()
resources_by_case: dict[str, list[dict[str, Any]]] = {}
all_resources: list[dict[str, Any]] = []

for case in DEMO_CASES:
    case_resources = [
        build_patient(case),
        build_condition(case),
        build_imaging_study(case, "baseline"),
        build_imaging_study(case, "followup"),
        build_prior_volume_observation(case),
    ]
    resources_by_case[case["case_id"]] = case_resources
    all_resources.extend(case_resources)

resource_type_counts = Counter(
    resource["resourceType"] for resource in all_resources
)

expected_counts = {
    "Patient": 3,
    "Condition": 3,
    "ImagingStudy": 6,
    "Observation": 3,
}
if dict(resource_type_counts) != expected_counts:
    raise AssertionError(
        f"Unexpected resource counts: {dict(resource_type_counts)}"
    )

print("=" * 88)
print("✅ Synthetic FHIR resources generated in memory")
print(f"✅ Total resources: {len(all_resources)}")
for resource_type in ("Patient", "Condition", "ImagingStudy", "Observation"):
    print(f"   {resource_type}: {resource_type_counts[resource_type]}")
print("✅ No current AI-generated follow-up Observation was created")
print("=" * 88)

✅ Synthetic FHIR resources generated in memory
✅ Total resources: 15
   Patient: 3
   Condition: 3
   ImagingStudy: 6
   Observation: 3
✅ No current AI-generated follow-up Observation was created


In [6]:
# Cell 6 — Validate resource structure, safety rules, and reference integrity

REQUIRED_TOP_LEVEL_FIELDS: dict[str, set[str]] = {
    "Patient": {"resourceType", "id", "meta", "identifier", "active", "name"},
    "Condition": {
        "resourceType",
        "id",
        "meta",
        "clinicalStatus",
        "verificationStatus",
        "code",
        "subject",
    },
    "ImagingStudy": {
        "resourceType",
        "id",
        "meta",
        "identifier",
        "status",
        "modality",
        "subject",
        "started",
    },
    "Observation": {
        "resourceType",
        "id",
        "meta",
        "status",
        "code",
        "subject",
        "effectiveDateTime",
        "valueQuantity",
        "derivedFrom",
    },
}

def collect_references(value: Any) -> list[str]:
    references: list[str] = []
    if isinstance(value, dict):
        for key, nested_value in value.items():
            if key == "reference" and isinstance(nested_value, str):
                references.append(nested_value)
            else:
                references.extend(collect_references(nested_value))
    elif isinstance(value, list):
        for item in value:
            references.extend(collect_references(item))
    return references

generated_reference_targets = {
    f"{resource['resourceType']}/{resource['id']}"
    for resource in all_resources
}

validation_errors: list[str] = []
validation_warnings: list[str] = []
reference_records: list[dict[str, Any]] = []

seen_resource_keys: set[str] = set()

for resource in all_resources:
    resource_type = resource.get("resourceType")
    resource_id = resource.get("id")
    resource_key = f"{resource_type}/{resource_id}"

    if resource_type not in REQUIRED_TOP_LEVEL_FIELDS:
        validation_errors.append(
            f"{resource_key}: unexpected resource type."
        )
        continue

    missing_fields = REQUIRED_TOP_LEVEL_FIELDS[resource_type] - set(resource)
    if missing_fields:
        validation_errors.append(
            f"{resource_key}: missing fields {sorted(missing_fields)}."
        )

    if not isinstance(resource_id, str) or not FHIR_ID_PATTERN.fullmatch(resource_id):
        validation_errors.append(f"{resource_key}: invalid FHIR id.")

    if resource_key in seen_resource_keys:
        validation_errors.append(f"{resource_key}: duplicate resource.")
    seen_resource_keys.add(resource_key)

    tags = resource.get("meta", {}).get("tag", [])
    has_synthetic_tag = any(
        tag.get("system") == DEMO_TAG_SYSTEM
        and tag.get("code") == "synthetic"
        for tag in tags
        if isinstance(tag, dict)
    )
    if not has_synthetic_tag:
        validation_errors.append(
            f"{resource_key}: missing required synthetic-data tag."
        )

    if resource_type == "Patient":
        if "address" in resource or "telecom" in resource:
            validation_errors.append(
                f"{resource_key}: address/telecom must not be included."
            )

    if resource_type == "ImagingStudy":
        if resource.get("status") != "available":
            validation_errors.append(
                f"{resource_key}: ImagingStudy.status must be available."
            )
        if not resource.get("series"):
            validation_errors.append(
                f"{resource_key}: at least one synthetic series is required."
            )

    if resource_type == "Observation":
        if resource.get("status") != "final":
            validation_errors.append(
                f"{resource_key}: prior historical Observation must be final."
            )
        note_text = " ".join(
            str(item.get("text", ""))
            for item in resource.get("note", [])
            if isinstance(item, dict)
        ).lower()
        if "prior human-reviewed historical measurement" not in note_text:
            validation_errors.append(
                f"{resource_key}: prior-reviewed status is not documented."
            )
        quantity = resource.get("valueQuantity", {})
        if quantity.get("system") != "http://unitsofmeasure.org":
            validation_errors.append(
                f"{resource_key}: UCUM system is required."
            )
        if quantity.get("code") != "mL":
            validation_errors.append(
                f"{resource_key}: volume unit must be mL."
            )
        if float(quantity.get("value", 0)) <= 0:
            validation_errors.append(
                f"{resource_key}: volume must be positive."
            )

    for reference in collect_references(resource):
        resolved = reference in generated_reference_targets
        reference_records.append(
            {
                "source": resource_key,
                "reference": reference,
                "resolved": resolved,
            }
        )
        if not resolved:
            validation_errors.append(
                f"{resource_key}: unresolved reference {reference!r}."
            )

# Case-level relationship and timeline checks
for case in DEMO_CASES:
    case_id = case["case_id"]
    case_resources = resources_by_case[case_id]
    case_counts = Counter(
        resource["resourceType"] for resource in case_resources
    )
    if dict(case_counts) != {
        "Patient": 1,
        "Condition": 1,
        "ImagingStudy": 2,
        "Observation": 1,
    }:
        validation_errors.append(
            f"{case_id}: incorrect case-level resource counts {dict(case_counts)}."
        )

    patient_reference = f"Patient/{case['patient']['id']}"
    for resource in case_resources:
        if resource["resourceType"] == "Patient":
            continue
        subject_reference = resource.get("subject", {}).get("reference")
        if subject_reference != patient_reference:
            validation_errors.append(
                f"{resource['resourceType']}/{resource['id']}: "
                "incorrect patient subject reference."
            )

    observation = next(
        resource
        for resource in case_resources
        if resource["resourceType"] == "Observation"
    )
    expected_baseline_reference = (
        f"ImagingStudy/imaging-baseline-{case_id}"
    )
    actual_derived_references = {
        item.get("reference")
        for item in observation.get("derivedFrom", [])
        if isinstance(item, dict)
    }
    if expected_baseline_reference not in actual_derived_references:
        validation_errors.append(
            f"{case_id}: prior Observation is not linked to baseline ImagingStudy."
        )

    if not (
        parse_fhir_datetime(case["baseline_datetime"])
        < parse_fhir_datetime(case["followup_datetime"])
    ):
        validation_errors.append(
            f"{case_id}: baseline/follow-up chronology is invalid."
        )

    recalculated_change = calculate_percent_change(
        float(case["baseline_volume_ml"]),
        float(case["planned_followup_reference_volume_ml"]),
    )
    if not math.isclose(
        recalculated_change,
        float(case["planned_percent_change"]),
        abs_tol=0.01,
    ):
        validation_errors.append(
            f"{case_id}: planned percentage change is inconsistent."
        )

if any(
    resource["resourceType"] == "Observation"
    and resource["id"].startswith("current-")
    for resource in all_resources
):
    validation_errors.append(
        "A current AI-generated Observation was created prematurely."
    )

resolved_reference_count = sum(
    1 for record in reference_records if record["resolved"]
)
reference_integrity_rate = (
    resolved_reference_count / len(reference_records)
    if reference_records
    else 1.0
)

validation_summary = {
    "validated_utc": utc_now(),
    "resource_count": len(all_resources),
    "resource_type_counts": dict(resource_type_counts),
    "reference_count": len(reference_records),
    "resolved_reference_count": resolved_reference_count,
    "reference_integrity_rate": round(reference_integrity_rate, 6),
    "errors": validation_errors,
    "warnings": validation_warnings,
    "passed": len(validation_errors) == 0,
}

if validation_errors:
    raise AssertionError(
        "Synthetic FHIR validation failed:\n"
        + "\n".join(f" - {error}" for error in validation_errors)
    )

print("=" * 88)
print("✅ Structural validation passed")
print("✅ Synthetic-data tags validated")
print("✅ Prior-reviewed Observation rule validated")
print("✅ No current AI result was created")
print(
    "✅ Reference integrity: "
    f"{resolved_reference_count}/{len(reference_records)} "
    f"({reference_integrity_rate:.1%})"
)
print("=" * 88)

✅ Structural validation passed
✅ Synthetic-data tags validated
✅ Prior-reviewed Observation rule validated
✅ No current AI result was created
✅ Reference integrity: 24/24 (100.0%)


## Output organization

Notebook 01 writes into a dedicated, rerunnable directory:

`data/synthetic_fhir/notebook_01/`

The output is separated by FHIR resource type and includes a non-FHIR case manifest.  
The case manifest contains planned later-test expectations, but it is **not** a clinical result and is **not** written to a FHIR server.

A transaction Bundle is not created here because write-back behavior and atomic transaction evidence belong to the later FHIR evidence/write-back notebook.

In [7]:
# Cell 7 — Write synthetic resources, case manifests, indexes, and documentation
import textwrap  # local defensive import

OUTPUT_ROOT = PROJECT_ROOT / "data/synthetic_fhir/notebook_01"
RESOURCE_DIRECTORIES = {
    "Patient": OUTPUT_ROOT / "patients",
    "Condition": OUTPUT_ROOT / "conditions",
    "ImagingStudy": OUTPUT_ROOT / "imaging_studies",
    "Observation": OUTPUT_ROOT / "observations",
}
CASE_PACKAGE_DIR = OUTPUT_ROOT / "case_packages"

for directory in [OUTPUT_ROOT, CASE_PACKAGE_DIR, *RESOURCE_DIRECTORIES.values()]:
    directory.mkdir(parents=True, exist_ok=True)

# Remove only previously generated Notebook 01 files, preserving unrelated project data.
for directory in [CASE_PACKAGE_DIR, *RESOURCE_DIRECTORIES.values()]:
    for stale_file in directory.glob("*.json"):
        stale_file.unlink()

resource_index: list[dict[str, Any]] = []
case_resource_files: dict[str, list[str]] = {
    case["case_id"]: [] for case in DEMO_CASES
}

for case_id, case_resources in resources_by_case.items():
    for resource in case_resources:
        resource_type = resource["resourceType"]
        resource_id = resource["id"]
        resource_path = (
            RESOURCE_DIRECTORIES[resource_type]
            / f"{resource_type.lower()}-{resource_id}.json"
        )
        write_json(resource_path, resource)
        relative_path = resource_path.relative_to(PROJECT_ROOT).as_posix()
        case_resource_files[case_id].append(relative_path)
        resource_index.append(
            {
                "case_id": case_id,
                "resource_type": resource_type,
                "resource_id": resource_id,
                "reference": f"{resource_type}/{resource_id}",
                "relative_path": relative_path,
            }
        )

demo_case_manifest = {
    "project_name": project_config["project_name"],
    "project_version": PROJECT_VERSION,
    "fhir_version": FHIR_VERSION,
    "generated_utc": generated_utc,
    "data_policy": {
        "synthetic_fhir_only": True,
        "real_patient_data_allowed": False,
        "public_deidentified_imaging_only": True,
    },
    "important_interpretation": (
        "Planned follow-up reference volumes are scenario-design values for "
        "later pipeline testing. They are not current AI-generated clinical results."
    ),
    "cases": [],
}

for case in DEMO_CASES:
    case_id = case["case_id"]
    case_manifest_entry = {
        "case_id": case_id,
        "patient_reference": f"Patient/{case['patient']['id']}",
        "condition_reference": f"Condition/condition-{case_id}",
        "baseline_imaging_reference": (
            f"ImagingStudy/imaging-baseline-{case_id}"
        ),
        "followup_imaging_reference": (
            f"ImagingStudy/imaging-followup-{case_id}"
        ),
        "prior_observation_reference": (
            f"Observation/prior-volume-{case_id}"
        ),
        "baseline_datetime": case["baseline_datetime"],
        "followup_datetime": case["followup_datetime"],
        "baseline_volume_ml": case["baseline_volume_ml"],
        "planned_followup_reference_volume_ml": (
            case["planned_followup_reference_volume_ml"]
        ),
        "planned_percent_change": case["planned_percent_change"],
        "planned_change_category": case["planned_change_category"],
        "future_qc_expectation": case["future_qc_expectation"],
        "future_review_expectation": case["future_review_expectation"],
        "scenario_description": case["scenario_description"],
        "resource_files": sorted(case_resource_files[case_id]),
    }
    demo_case_manifest["cases"].append(case_manifest_entry)
    write_json(
        CASE_PACKAGE_DIR / f"{case_id}_case_context.json",
        case_manifest_entry,
    )

DEMO_CASE_MANIFEST_PATH = OUTPUT_ROOT / "demo_case_manifest.json"
RESOURCE_INDEX_JSON_PATH = OUTPUT_ROOT / "resource_index.json"
RESOURCE_INDEX_CSV_PATH = OUTPUT_ROOT / "resource_index.csv"
REFERENCE_REPORT_PATH = OUTPUT_ROOT / "reference_integrity.json"
OUTPUT_README_PATH = OUTPUT_ROOT / "README.md"

write_json(DEMO_CASE_MANIFEST_PATH, demo_case_manifest)
write_json(
    RESOURCE_INDEX_JSON_PATH,
    {
        "generated_utc": generated_utc,
        "resource_count": len(resource_index),
        "resources": sorted(
            resource_index,
            key=lambda row: (
                row["case_id"],
                row["resource_type"],
                row["resource_id"],
            ),
        ),
    },
)
write_json(
    REFERENCE_REPORT_PATH,
    {
        "generated_utc": generated_utc,
        "reference_integrity_rate": reference_integrity_rate,
        "reference_count": len(reference_records),
        "resolved_reference_count": resolved_reference_count,
        "references": reference_records,
    },
)

with RESOURCE_INDEX_CSV_PATH.open(
    "w",
    encoding="utf-8",
    newline="",
) as handle:
    writer = csv.DictWriter(
        handle,
        fieldnames=[
            "case_id",
            "resource_type",
            "resource_id",
            "reference",
            "relative_path",
        ],
    )
    writer.writeheader()
    writer.writerows(
        sorted(
            resource_index,
            key=lambda row: (
                row["case_id"],
                row["resource_type"],
                row["resource_id"],
            ),
        )
    )

output_readme = f"""# Notebook 01 Synthetic FHIR Foundation

Generated by `{NOTEBOOK_FILENAME}`.

## Scope

This directory contains synthetic FHIR R4 patient and longitudinal imaging context for the
locked NeuroFHIR-QC demonstration cases: stable, progression, and low-confidence.

## Generated FHIR resources

- 3 Patient resources
- 3 Condition resources
- 6 ImagingStudy resources
- 3 prior reviewed baseline-volume Observation resources

## Safety interpretation

- All records are synthetic.
- No real patient data are present.
- The prior Observations are historical synthetic reviewed measurements.
- No current AI-generated follow-up Observation is created in Notebook 01.
- No transaction Bundle or server write-back is performed in Notebook 01.
- Planned follow-up reference values in `demo_case_manifest.json` are test-scenario
  design values, not clinical results.

## Main files

- `demo_case_manifest.json`: planned case behavior for later notebooks
- `resource_index.json` and `resource_index.csv`: generated-resource inventory
- `reference_integrity.json`: internal FHIR-reference resolution evidence
- resource-type subdirectories: individual FHIR JSON files
- `case_packages/`: non-FHIR case-context indexes
"""
OUTPUT_README_PATH.write_text(
    textwrap.dedent(output_readme).strip() + "\n",
    encoding="utf-8",
)

expected_output_files = [
    DEMO_CASE_MANIFEST_PATH,
    RESOURCE_INDEX_JSON_PATH,
    RESOURCE_INDEX_CSV_PATH,
    REFERENCE_REPORT_PATH,
    OUTPUT_README_PATH,
    *[
        Path(PROJECT_ROOT / row["relative_path"])
        for row in resource_index
    ],
    *sorted(CASE_PACKAGE_DIR.glob("*.json")),
]

empty_or_missing_outputs = [
    str(path)
    for path in expected_output_files
    if not path.exists() or path.stat().st_size == 0
]
if empty_or_missing_outputs:
    raise FileNotFoundError(
        "Expected Notebook 01 outputs are missing or empty:\n"
        + "\n".join(f" - {path}" for path in empty_or_missing_outputs)
    )

print("=" * 88)
print("✅ Synthetic FHIR files written successfully")
print(f"📁 Output root: {OUTPUT_ROOT}")
print(f"📄 Individual FHIR resources: {len(resource_index)}")
print(f"📦 Case-context indexes: {len(list(CASE_PACKAGE_DIR.glob('*.json')))}")
print(f"📋 Demo-case manifest: {DEMO_CASE_MANIFEST_PATH}")
print(f"🔗 Reference report: {REFERENCE_REPORT_PATH}")
print("=" * 88)

✅ Synthetic FHIR files written successfully
📁 Output root: /content/drive/MyDrive/neurofhir-qc/data/synthetic_fhir/notebook_01
📄 Individual FHIR resources: 15
📦 Case-context indexes: 3
📋 Demo-case manifest: /content/drive/MyDrive/neurofhir-qc/data/synthetic_fhir/notebook_01/demo_case_manifest.json
🔗 Reference report: /content/drive/MyDrive/neurofhir-qc/data/synthetic_fhir/notebook_01/reference_integrity.json


In [8]:
# Cell 8 — Create checksums, the Notebook 01 audit, and a human-readable execution record

AUDIT_JSON_PATH = (
    PROJECT_ROOT
    / "evaluation/results/notebook_01_synthetic_fhir_audit.json"
)
AUDIT_MD_PATH = (
    PROJECT_ROOT
    / "docs/NOTEBOOK_01_SYNTHETIC_FHIR_FOUNDATION.md"
)

generated_output_files = sorted(
    {
        path.resolve()
        for path in expected_output_files
        if path.exists() and path.is_file()
    },
    key=lambda path: str(path),
)

checksum_inventory = [
    {
        "relative_path": path.relative_to(PROJECT_ROOT).as_posix(),
        "size_bytes": path.stat().st_size,
        "sha256": sha256_file(path),
    }
    for path in generated_output_files
]

notebook_saved = NOTEBOOK_SAVE_PATH.exists() and NOTEBOOK_SAVE_PATH.stat().st_size > 0
manifest_status = "completed" if notebook_saved else "executed_pending_notebook_save"

notebook_01_entry["status"] = manifest_status
notebook_01_entry["last_executed_utc"] = utc_now()
notebook_01_entry["generated_resource_count"] = len(all_resources)
notebook_01_entry["audit_path"] = AUDIT_JSON_PATH.relative_to(PROJECT_ROOT).as_posix()
notebook_01_entry["completion_note"] = (
    "Execution and outputs passed; notebook file is present in the project notebooks directory."
    if notebook_saved
    else (
        "Execution and outputs passed, but the notebook file has not yet been "
        "confirmed in the project notebooks directory. Save it there and rerun Cell 8."
    )
)

write_json(NOTEBOOK_MANIFEST_PATH, notebook_manifest)

audit_payload = {
    "project_name": project_config["project_name"],
    "project_version": PROJECT_VERSION,
    "notebook_number": NOTEBOOK_NUMBER,
    "notebook_filename": NOTEBOOK_FILENAME,
    "generated_utc": generated_utc,
    "audited_utc": utc_now(),
    "status": manifest_status,
    "notebook_saved_to_project": notebook_saved,
    "required_notebook_save_path": str(NOTEBOOK_SAVE_PATH),
    "foundation_gate": {
        "notebook_00_manifest_status": notebook_00_status,
        "foundation_audit_json": str(FOUNDATION_AUDIT_JSON),
        "foundation_audit_markdown": str(FOUNDATION_AUDIT_MD),
        "passed": True,
    },
    "scope": {
        "demo_cases": case_ids,
        "resource_types_created": sorted(resource_type_counts),
        "resource_type_counts": dict(resource_type_counts),
        "current_ai_observation_created": False,
        "hapi_server_used": False,
        "transaction_bundle_created": False,
        "fhir_writeback_performed": False,
    },
    "data_policy": {
        "synthetic_fhir_only": True,
        "real_patient_data_allowed": False,
        "public_deidentified_imaging_only": True,
    },
    "validation": validation_summary,
    "output_root": str(OUTPUT_ROOT),
    "output_file_count": len(checksum_inventory),
    "checksum_inventory": checksum_inventory,
    "next_notebook": (
        "02 — HAPI Server and Read Operations, only after this notebook "
        "is marked completed."
    ),
}
write_json(AUDIT_JSON_PATH, audit_payload)

case_summary_lines = []
for case in DEMO_CASES:
    case_summary_lines.append(
        f"| {case['case_id']} | {case['patient']['id']} | "
        f"{case['baseline_volume_ml']:.1f} mL | "
        f"{case['planned_followup_reference_volume_ml']:.1f} mL | "
        f"{case['planned_percent_change']:+.2f}% | "
        f"{case['future_qc_expectation']} |"
    )

audit_markdown = f"""# Notebook 01 — Synthetic FHIR Foundation Execution Record

**Notebook:** `{NOTEBOOK_FILENAME}`
**Project:** NeuroFHIR-QC
**FHIR version:** R4
**Execution status:** `{manifest_status}`
**Generated UTC:** {generated_utc}

## Completed work

Notebook 01 generated deterministic synthetic longitudinal FHIR context for the three
locked demonstration cases.

| Resource type | Count |
|---|---:|
| Patient | {resource_type_counts['Patient']} |
| Condition | {resource_type_counts['Condition']} |
| ImagingStudy | {resource_type_counts['ImagingStudy']} |
| Prior baseline Observation | {resource_type_counts['Observation']} |
| **Total** | **{len(all_resources)}** |

## Case design

| Case | Synthetic patient id | Baseline | Planned later-test reference | Planned change | Future QC expectation |
|---|---|---:|---:|---:|---|
{chr(10).join(case_summary_lines)}

Planned follow-up values are scenario-design values for later testing. They are not
current AI-generated results.

## Validation evidence

- Structural validation passed: **yes**
- Synthetic-data tags present: **yes**
- Internal references resolved: **{resolved_reference_count}/{len(reference_records)}**
- Reference-integrity rate: **{reference_integrity_rate:.1%}**
- Current AI-generated Observation created: **no**
- HAPI server used: **no**
- FHIR write-back performed: **no**
- Transaction Bundle created: **no**

## Output locations

- Synthetic FHIR root: `{OUTPUT_ROOT}`
- Demo-case manifest: `{DEMO_CASE_MANIFEST_PATH}`
- Resource index: `{RESOURCE_INDEX_JSON_PATH}`
- Reference-integrity report: `{REFERENCE_REPORT_PATH}`
- Machine-readable audit: `{AUDIT_JSON_PATH}`

## Completion gate

Required saved-notebook path:

`{NOTEBOOK_SAVE_PATH}`

Notebook file detected at that path: **{'yes' if notebook_saved else 'no'}**

If the answer is no, save the notebook into the path above and rerun Cell 8. The manifest
intentionally remains `executed_pending_notebook_save` until that evidence exists.

## What remains unbuilt

This notebook did not connect to HAPI FHIR, implement SMART launch, download MRI,
run segmentation, calculate QC, execute human-review transitions, or perform FHIR
transaction write-back.
"""

AUDIT_MD_PATH.parent.mkdir(parents=True, exist_ok=True)
AUDIT_MD_PATH.write_text(
    textwrap.dedent(audit_markdown).strip() + "\n",
    encoding="utf-8",
)

# Recompute the audit checksum after the final audit files are present.
final_required_files = [
    AUDIT_JSON_PATH,
    AUDIT_MD_PATH,
    NOTEBOOK_MANIFEST_PATH,
    *generated_output_files,
]
final_missing_or_empty = [
    str(path)
    for path in final_required_files
    if not path.exists() or path.stat().st_size == 0
]
if final_missing_or_empty:
    raise FileNotFoundError(
        "Notebook 01 final audit files are missing or empty:\n"
        + "\n".join(f" - {path}" for path in final_missing_or_empty)
    )

print("=" * 88)
print("✅ Notebook 01 resource generation passed")
print("✅ Structural and reference-integrity checks passed")
print(f"✅ Audit JSON: {AUDIT_JSON_PATH}")
print(f"✅ Audit Markdown: {AUDIT_MD_PATH}")
print(f"📓 Notebook manifest status: {manifest_status}")
if notebook_saved:
    print("✅ Notebook file detected in the project notebooks directory")
    print("🎯 Notebook 01 is complete; Notebook 02 may begin")
else:
    print("⚠️ Save this notebook to the required project path, then rerun Cell 8")
    print(f"   Required path: {NOTEBOOK_SAVE_PATH}")
    print("⛔ Do not begin Notebook 02 until the manifest status becomes completed")
print("=" * 88)

✅ Notebook 01 resource generation passed
✅ Structural and reference-integrity checks passed
✅ Audit JSON: /content/drive/MyDrive/neurofhir-qc/evaluation/results/notebook_01_synthetic_fhir_audit.json
✅ Audit Markdown: /content/drive/MyDrive/neurofhir-qc/docs/NOTEBOOK_01_SYNTHETIC_FHIR_FOUNDATION.md
📓 Notebook manifest status: executed_pending_notebook_save
⚠️ Save this notebook to the required project path, then rerun Cell 8
   Required path: /content/drive/MyDrive/neurofhir-qc/notebooks/01_NeuroFHIR_QC_Synthetic_FHIR_Foundation.ipynb
⛔ Do not begin Notebook 02 until the manifest status becomes completed
